# 📖 Tools and Function Calling

---

## 🎯 Learning Objectives

By the end of this notebook, you will:

1. **Understand** function calling as tool use
2. **Create** custom tools with proper schemas
3. **Handle** tool outputs and iterations
4. **Debug** tool-related issues

---

## ⏱️ Time Estimate

**~30 minutes**

## 📦 Setup

In [ ]:
!pip install -q openai langchain langchain-core
import os
if "OPENAI_API_KEY" not in os.environ:
    import getpass
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OpenAI API Key: ")

from openai import OpenAI
client = OpenAI()

---

## 🧠 Theory: Function Calling = Tools

When you use OpenAI's function calling (or tool calling), you're essentially giving the LLM **tools** to interact with the world.

```
┌─────────────────────────────────────────────────────┐
│           FUNCTION CALLING = TOOLS                 │
├─────────────────────────────────────────────────────┤
│                                                     │
│  1. Define function with JSON schema               │
│  2. Send to LLM with available tools               │
│  3. LLM decides to call a function                 │
│  4. Execute function with parameters                │
│  5. Return result to LLM                          │
│  6. LLM generates final response                   │
│                                                     │
└─────────────────────────────────────────────────────┘
```

### How Tool Calling Works

The LLM doesn't run the function directly - it **decides** to call it and outputs the parameters. Your code executes the function.

```
User: What's the weather in Tokyo?

LLM Decision: I need weather data. Let me call the get_weather tool.
LLM Output (to API): {
  tool_calls: [{
    name: "get_weather",
    arguments: {"location": "Tokyo"}
  }]
}

Your Code: Executes get_weather(location="Tokyo")
Returns: {"temperature": 22, "conditions": "sunny"}

LLM Final: The weather in Tokyo is sunny with 22°C.

---

## 💻 Creating Tools with LangChain

Let's create a custom tool:

In [ ]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

# Method 1: Using the @tool decorator
@tool
def get_stock_price(symbol: str) -> str:
    """Get the current stock price for a given symbol.
    
    Args:
        symbol: Stock ticker symbol (e.g., 'AAPL', 'GOOGL')
    
    Returns:
        The current stock price with context
    """
    # In real implementation, you'd call a stock API here
    mock_prices = {
        "AAPL": "$178.50",
        "GOOGL": "$142.30",
        "MSFT": "$378.20",
        "TSLA": "$245.80"
    }
    price = mock_prices.get(symbol.upper(), "$0.00")
    return f"{symbol.upper()} is currently trading at {price}"

print("✅ Tool created: get_stock_price")
print(f"   Name: {get_stock_price.name}")
print(f"   Description: {get_stock_price.description}")

In [ ]:
# Method 2: Using Tool.from_function
def fetch_crypto_price(coin: str) -> str:
    """Fetch the current price of a cryptocurrency.
    
    Args:
        coin: Cryptocurrency name (bitcoin, ethereum, etc.)
    
    Returns:
        The current price in USD
    """
    mock_prices = {
        "bitcoin": "$52,340",
        "ethereum": "$2,890",
        "solana": "$98.50"
    }
    price = mock_prices.get(coin.lower(), "$0.00")
    return f"{coin.capitalize()} is currently trading at {price}"

# Create tool from function
from langchain_core.tools import Tool

crypto_tool = Tool.from_function(
    func=fetch_crypto_price,
    name="crypto_price",
    description="Get cryptocurrency prices. Input should be 'bitcoin', 'ethereum', or 'solana'."
)

print("✅ Tool created: crypto_price")
print(f"   Name: {crypto_tool.name}")
print(f"   Description: {crypto_tool.description}")

---

## 🔧 Tool Schema Deep Dive

Let's see the auto-generated JSON schema:

In [ ]:
import json

# View the auto-generated schema
print("🔍 Tool Schema")
print("=" * 60)
print(json.dumps(get_stock_price.args_schema.schema(), indent=2))

### Schema Best Practices

| Field | Purpose | Tip |
|-------|---------|-----|
**| description | Explains what the tool does | Be clear and specific! |
**| args | Defines parameters | Include examples |
**| name | Tool identifier | Use snake_case |

**Pro tip**: The description is CRITICAL - the LLM uses it to decide WHEN to call the tool.

---

## 🔨 Binding Tools to an LLM

Now let's bind tools to an LLM and make calls:

In [ ]:
# Create LLM with tools bound
llm = ChatOpenAI(model="gpt-4o-mini")
llm_with_tools = llm.bind_tools([get_stock_price, crypto_tool])

# Test 1: Ask about stocks (should use tool)
from langchain_core.messages import HumanMessage

response1 = llm_with_tools.invoke("What's the price of AAPL?")
print("📨 Test 1: Stock price query")
print(f"   Tool calls: {response1.tool_calls}")
print(f"   Content: {response1.content}")
print()

# Test 2: Ask about crypto (should use tool)
response2 = llm_with_tools.invoke("How much is Bitcoin worth?")
print("📨 Test 2: Crypto query")
print(f"   Tool calls: {response2.tool_calls}")
print(f"   Content: {response2.content}")
print()

# Test 3: Ask something else (no tool needed)
response3 = llm_with_tools.invoke("Hello! How are you?")
print("📨 Test 3: No tool needed")
print(f"   Tool calls: {response3.tool_calls}")
print(f"   Content: {response3.content}")

### Key Insight: When to Call Tools

The LLM decides whether to call a tool based on the **tool descriptions** provided. 

| Question | Tool Called? | Why? |
|-----------|-------------|------|
**| "What's AAPL?" | ✅ Yes | Stock price tool |
**| "Hello!" | ❌ No | Conversational |
**| "Tell me about AI" | ❌ No | Not a tool task |

---

## 🔄 Handling Tool Results

Let's create a complete agent with tool handling:

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

# Simple agent with tool execution
def run_agent_with_tools(user_input, max_iterations=5):
    """Run a simple agent that can use tools."""
    
    # Initialize
    llm = ChatOpenAI(model="gpt-4o-mini")
    llm_with_tools = llm.bind_tools([get_stock_price, crypto_tool])
    
    messages = [HumanMessage(content=user_input)]
    
    for i in range(max_iterations):
        print(f"\n📍 Iteration {i+1}")
        
        # Get LLM response
        response = llm_with_tools.invoke(messages)
        messages.append(response)
        
        # Check if tool called
        if response.tool_calls:
            print(f"   🔧 Tool called: {[tc['name'] for tc in response.tool_calls]}")
            
            # Execute tools
            for tool_call in response.tool_calls:
                tool_name = tool_call['name']
                tool_args = tool_call['arguments']
                
                # Find and execute tool
                if tool_name == "get_stock_price":
                    result = get_stock_price.invoke(tool_args)
                elif tool_name == "crypto_price":
                    result = crypto_tool.invoke(tool_args)
                else:
                    result = "Unknown tool"
                
                print(f"   📦 Result: {result}")
                messages.append(HumanMessage(content=str(result)))
        else:
            print(f"   ✅ Final response: {response.content}")
            return response.content
    
    return "Max iterations reached"

# Run the agent
result = run_agent_with_tools("What's the price of GOOGL?")

---

## ⚠️ Common Tool Calling Issues

Let's look at problems and solutions:

In [ ]:
# Issue 1: Bad tool description leads to wrong tool
print("⚠️ Issue 1: Ambiguous descriptions")
print("=" * 60)

# Tool with BAD description
@tool
def bad_weather_tool(location: str) -> str:
    """Get weather."""
    return "Sunny"

@tool  
def bad_stock_tool(symbol: str) -> str:
    """Get stock info."""
    return "$100"

# The LLM might get confused!
llm = ChatOpenAI(model="gpt-4o-mini")
llm_bad = llm.bind_tools([bad_weather_tool, bad_stock_tool])
response = llm_bad.invoke("What's the weather in New York?")
print(f"Bad descriptions → Tool: {response.tool_calls[0]['name'] if response.tool_calls else 'None'}")

# Tool with GOOD description
@tool
def good_weather_tool(location: str) -> str:
    """Get current weather conditions for a city.
    
    Args:
        location: City name (e.g., 'New York', 'London')
    
    Returns:
        Weather description (sunny, rainy, etc.)
    """
    return "Sunny"

llm_good = llm.bind_tools([good_weather_tool])
response = llm_good.invoke("What's the weather in New York?")
print(f"Good descriptions → Tool: {response.tool_calls[0]['name'] if response.tool_calls else 'None'}")

### Tool Debugging Checklist

```
┌─────────────────────────────────────────────────────┐
│           TOOL CALLING DEBUGGING                      │
├─────────────────────────────────────────────────────┤
│  ✅ Description is clear and specific?             │
│  ✅ Args defined with types?                      │
│  ✅ Examples in description?                     │
│  ✅ Tool name is descriptive?                    │
│  ❌ Is another tool too similar?                  │
└─────────────────────────────────────────────────────┘
```

---

## 🧪 Try It Yourself!

**Exercise 1**: Create a tool that converts currency (USD to EUR).

**Exercise 2**: Create two tools and test which one gets called.

**Exercise 3**: Simulate a tool that fails and handle the error.

In [ ]:
# 🧪 Exercise 1: Create a currency converter tool

@tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert one currency to another.
    
    Args:
        amount: Amount to convert
        from_currency: Source currency (USD, EUR, GBP)
        to_currency: Target currency (USD, EUR, GBP)
    
    Returns:
        Converted amount
    """
    rates = {"USD": 1, "EUR": 0.92, "GBP": 0.79}
    in_usd = amount / rates[from_currency]
    result = in_usd * rates[to_currency]
    return f"{amount} {from_currency} = {result:.2f} {to_currency}"

# Test manually
print("Testing:", convert_currency.invoke({"amount": 100, "from_currency": "USD", "to_currency": "EUR"}))

---

## ❓ FAQ

**Q: Can I call multiple tools at once?**
A: Yes! The model can return multiple tool_calls in a single response.

**Q: What if the tool fails?**
A: Return the error to the LLM so it can decide next steps.

**Q: Do tools cost money?**
A: Tools themselves are free, but LLM API calls cost as normal.

**Q: Can tools call other tools?**
A: In principle yes, but this creates complex chains - covered in LangGraph!

---

## ✅ Summary

You now understand:

1. **Function calling** = tool use by the LLM
2. **Tool schema** = JSON defining what tool does
3. **Binding tools** = connecting tools to LLM
4. **Handling tool calls** = executing and returning results
5. **Debugging** = checking descriptions and schemas

**Key Insight**: Tools are what turn a static LLM into an ACTIVE agent!

---

## 🔗 Next Steps

Next: **[03_routers_and_skills.ipynb](03_routers_and_skills.ipynb)** - Learn how agents route between tasks!